# Wanderbricks Cancellation Predictor
## Databricks ML Overview — Training Session

**Business scenario:** You are a data scientist at Wanderbricks, a fictional travel booking platform. The customer operations team has flagged rising cancellation rates. Your job: build a model that predicts which new bookings are likely to cancel so the team can intervene early with reminders or incentives.

**What we cover:**
1. Explore data in the `samples` catalog
2. Engineer features using Apache Spark
3. Train a classification model with **MLflow autologging**
4. Review the MLflow Experiment UI
5. Evaluate the model
6. Save and reload the model using MLflow
7. Understand how CI/CD fits into the Databricks ML workflow

> **Workspace:** Databricks Individual (free) edition.  
> Steps marked **Paid feature** describe capabilities available in paid or trial workspaces.

In [ ]:
# All libraries are pre-installed in Databricks Runtime — no pip install needed
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, ConfusionMatrixDisplay

print('MLflow version:', mlflow.__version__)

---
## Section 1 — Explore the Data

We read directly from the `samples` catalog — no connection strings, no JDBC setup required.  
Unity Catalog governs access automatically.

**`display()` vs `.show()`:** Always use `display()` in Databricks notebooks. It renders an interactive table with column sorting, filtering, and a built-in chart builder.

In [ ]:
bookings_raw = spark.read.table('samples.wanderbricks.bookings')
display(bookings_raw)

In [ ]:
%sql
SELECT status, COUNT(*) AS booking_count
FROM samples.wanderbricks.bookings
GROUP BY status
ORDER BY booking_count DESC

In [ ]:
%sql
-- Run this first! Verify column names match what the feature engineering code expects.
-- Adjust column names in Section 2 if your schema differs.
DESCRIBE TABLE samples.wanderbricks.bookings

---
## Section 2 — Feature Engineering with Spark

We join four tables with `bookings` as the anchor, derive new columns using `pyspark.sql.functions`, then convert to Pandas for scikit-learn.

| Join | Purpose |
|---|---|
| `bookings → users` | User tenure, age, loyalty tier |
| `bookings → properties` | Property type and average rating |
| `bookings → payments` | Payment method and timing |

> **Why LEFT JOIN?** Not every booking has a completed payment record. Left joins preserve all bookings and let us handle nulls explicitly.

> **Why NOT join `reviews`?** Reviews are written *after* the stay. Using them to predict cancellation would be **data leakage** — the model would train on information it cannot have at prediction time.

In [ ]:
from pyspark.sql import functions as F

bookings   = spark.read.table('samples.wanderbricks.bookings')
users      = spark.read.table('samples.wanderbricks.users')
properties = spark.read.table('samples.wanderbricks.properties')
payments   = spark.read.table('samples.wanderbricks.payments')

print('Bookings  :', bookings.count())
print('Users     :', users.count())
print('Properties:', properties.count())
print('Payments  :', payments.count())

In [ ]:
# --- Join tables (bookings is the anchor) ---
df = (
    bookings
    .join(users.select('user_id', 'signup_date', 'age', 'loyalty_tier', 'country'),
          on='user_id', how='left')
    .join(properties.select('property_id', 'property_type', 'avg_rating'),
          on='property_id', how='left')
    .join(payments.select('booking_id', 'payment_method', 'payment_date'),
          on='booking_id', how='left')
)

# --- Derive features ---
df = (
    df
    # Target variable: 1 = cancelled, 0 = everything else
    .withColumn('is_cancelled',
                F.when(F.col('status') == 'cancelled', 1).otherwise(0))
    # Days between booking date and check-in (longer lead = higher cancel risk)
    .withColumn('lead_time_days',
                F.datediff(F.col('check_in_date'), F.col('booking_date')))
    # Length of the stay in nights
    .withColumn('length_of_stay',
                F.datediff(F.col('check_out_date'), F.col('check_in_date')))
    # Month of booking (captures seasonality)
    .withColumn('booking_month',
                F.month(F.col('booking_date')))
    # How long the user has been a member at time of booking
    .withColumn('user_tenure_days',
                F.datediff(F.col('booking_date'), F.col('signup_date')))
    # Days between booking and payment (delayed payment is a cancel signal)
    .withColumn('days_to_payment',
                F.datediff(F.col('payment_date'), F.col('booking_date')))
)

# --- Select final feature set ---
feature_cols = [
    'is_cancelled',
    'lead_time_days', 'length_of_stay', 'total_amount', 'num_guests', 'booking_month',
    'user_tenure_days', 'age',
    'loyalty_tier', 'country',
    'property_type', 'avg_rating',
    'payment_method', 'days_to_payment',
]

df_features = df.select(*feature_cols)
display(df_features.limit(10))

In [ ]:
# Convert to Pandas for scikit-learn
# Fine for a sample dataset; in production keep everything in Spark or use Spark MLlib
df_pd = df_features.toPandas()

print('Shape:', df_pd.shape)
print('Cancellation rate: {:.1%}'.format(df_pd['is_cancelled'].mean()))
print('\nNull counts:')
print(df_pd.isnull().sum())

In [ ]:
# One-hot encode categoricals
categorical_cols = ['loyalty_tier', 'property_type', 'payment_method', 'country']
df_model = pd.get_dummies(df_pd, columns=categorical_cols, drop_first=True)

# Fill nulls from left joins with 0
df_model = df_model.fillna(0)

X = df_model.drop('is_cancelled', axis=1)
y = df_model['is_cancelled']

# stratify=y ensures both splits have the same cancellation rate
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Training samples:', len(X_train), ' | Test samples:', len(X_test))
print('Cancellation rate in train: {:.1%}'.format(y_train.mean()))
print('Number of features:', X.shape[1])

---
## Section 3 — Train with MLflow Autologging

`mlflow.sklearn.autolog()` is the single most impactful Databricks ML feature for beginners.  
**One line** automatically captures:
- All model hyperparameters (`n_estimators`, `max_depth`, …)
- Training metrics (accuracy, F1, precision, recall)
- The serialised model artifact
- Feature importance plots

Without autolog, you would write 20+ `mlflow.log_param()` calls by hand.

---

**Why `class_weight='balanced'`?**  
Cancellations are a minority class — typically 10–20% of all bookings.  
Without this, the model learns to always predict *'not cancelled'* and still achieves 80%+ accuracy — without actually learning anything useful.

In [ ]:
# Group all experiment runs under one named experiment
mlflow.set_experiment('/Users/wanderbricks_cancellation')

# One line replaces 20+ mlflow.log_param() calls
mlflow.sklearn.autolog(log_input_examples=True, log_model_signatures=True)

with mlflow.start_run(run_name='rf_baseline_v1') as run:
    rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=8,
        class_weight='balanced',  # critical for imbalanced cancellation data
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)

    # Autolog captures training metrics; we manually add the held-out test AUC
    y_prob = rf.predict_proba(X_test)[:, 1]
    test_auc = roc_auc_score(y_test, y_prob)
    mlflow.log_metric('test_roc_auc', test_auc)

    run_id = run.info.run_id

print('Run ID :', run_id)
print('Test AUC: {:.3f}'.format(test_auc))

---
## Section 4 — MLflow Experiment UI Walkthrough

> **Live demo — no new code in this section.**

Navigate to the **Experiments** tab in the left sidebar → open **wanderbricks_cancellation**.

| Tab | What you see |
|---|---|
| **Parameters** | `n_estimators`, `max_depth`, `class_weight` — auto-captured |
| **Metrics** | accuracy, F1, precision, recall, `test_roc_auc` |
| **Artifacts** | Serialised model, `MLmodel` manifest, feature importance plot |

**Key insight:** This is your audit trail.  
Six months from now you will know exactly what data, what code, and what parameters produced this model — without maintaining a spreadsheet.

**Try this:** Run the training cell again with `n_estimators=50`, then click **Compare runs** in the UI to see the difference side by side.

---
## Section 5 — Evaluate the Model

Three visualisations — no more:

1. **AUC score** — primary metric for imbalanced classification
2. **Confusion matrix** — true/false positives and negatives at a glance
3. **Feature importance** — which signals actually drive the prediction

**Why AUC and not accuracy?**  
If 15% of bookings cancel, a model that always predicts *'no cancel'* achieves 85% accuracy — and is completely useless.  
AUC measures whether the model consistently ranks cancellations *above* non-cancellations, regardless of threshold.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
ConfusionMatrixDisplay.from_estimator(
    rf, X_test, y_test, ax=axes[0], colorbar=False
)
axes[0].set_title('Confusion Matrix')

# Feature importance — top 10
importance = pd.Series(rf.feature_importances_, index=X.columns)
importance.nlargest(10).sort_values().plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Top 10 Feature Importances')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

print('Test AUC: {:.3f}'.format(test_auc))

---
## Section 6 — Save and Load the Model with MLflow

On the Individual (free) edition, we load the model directly from the MLflow run URI — no separate model registry step required.

**What this looks like in a paid workspace:**
```python
mlflow.set_registry_uri('databricks-uc')
mlflow.register_model(model_uri, 'main.default.cancellation_predictor')

# Set a stable alias
client.set_registered_model_alias('main.default.cancellation_predictor', 'champion', version)

# Inference pipelines always reference the alias, not the version number
mlflow.pyfunc.load_model('models:/main.default.cancellation_predictor@champion')
```
When you promote a new model, you move the `@champion` alias. The inference pipeline code never changes.

In [ ]:
# Load the model directly from the MLflow run artifact (works on free edition)
model_uri = 'runs:/' + run_id + '/model'
loaded_model = mlflow.sklearn.load_model(model_uri)

# Score 5 test samples — proves the round-trip works
sample_input  = X_test.head(5)
predictions   = loaded_model.predict(sample_input)
probabilities = loaded_model.predict_proba(sample_input)[:, 1]

result = sample_input[['lead_time_days', 'total_amount']].copy()
result['predicted_cancelled']      = predictions
result['cancellation_probability'] = probabilities.round(3)

display(result)

---
## Section 7 — CI/CD in Databricks: Conceptual Overview

On the free edition, full CI/CD cannot be demonstrated live — but the pattern is straightforward:

```
Developer pushes notebook/code to GitHub
          |
          v
GitHub Actions triggers on PR or merge to main
          |
          v
databricks bundle deploy  (deploys code to staging workspace)
          |
          v
Databricks Workflow runs automated tests
(unit tests + model quality validation)
          |
          v
On success: model alias promoted to @champion in prod workspace
          |
          v
Inference pipeline always loads @champion -- no code changes needed
```

**Key tools:**

| Tool | Role |
|---|---|
| **Databricks Asset Bundles (DABs)** | Infrastructure-as-code — define notebooks, jobs, clusters in `databricks.yml` |
| **Databricks CLI** | Runs `databricks bundle deploy` from GitHub Actions |
| **MLflow Model Registry** | Model versioning, aliases, and lineage (Unity Catalog — paid feature) |
| **Databricks Workflows** | Orchestration for scheduled and event-triggered jobs |

> You can version this notebook in Git right now — connect via **Repos** in the workspace sidebar and commit/push to GitHub.  
> The multi-environment deployment step requires a trial or paid workspace.

---
## Summary

| Step | What we used | Why it matters |
|---|---|---|
| Data access | `spark.read.table()` + `samples` catalog | No connection strings; governed access |
| Feature engineering | PySpark `withColumn()`, `datediff()` | Scalable; same code works on billions of rows |
| Model training | `RandomForestClassifier` + `mlflow.sklearn.autolog()` | Full experiment tracking in one line |
| Experiment tracking | MLflow Experiments UI | Reproducible, auditable, comparable runs |
| Model saving | `mlflow.sklearn.load_model()` from run URI | Portable artifact; works on free edition |
| CI/CD | Databricks Asset Bundles + GitHub Actions | Automated, governed ML deployment |

**Try next:**
- Change `n_estimators` or `max_depth` and compare runs in the Experiment UI
- Remove `class_weight='balanced'` — watch how accuracy goes up but AUC drops
- Explore **Databricks AutoML** for automated feature engineering and model selection